In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
ZIP_PATH = "/content/drive/MyDrive/EdgeCard_System/dataset_rec.zip"

DATA_DIR = "/content/dataset_rec"

OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "EdgeCard_System/stage_3/pp-ocrv6_small_rec"
)

In [ ]:
import os
import shutil

if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)

!unzip -q "{ZIP_PATH}" -d /content/

print(os.listdir(DATA_DIR))

In [ ]:
# CÀI PaddlePaddle GPU

!python -m pip install -q \
    paddlepaddle-gpu==3.2.1 \
    -i https://www.paddlepaddle.org.cn/packages/stable/cu126/

In [ ]:
import paddle

print("Paddle :", paddle.__version__)
print("CUDA   :", paddle.version.cuda())
print("cuDNN  :", paddle.version.cudnn())
print("GPU build:", paddle.device.is_compiled_with_cuda())

paddle.utils.run_check()

In [ ]:
paddle.set_device("gpu")

x = paddle.randn([1024, 1024])
y = paddle.randn([1024, 1024])

z = paddle.matmul(x, y)

print("GPU computation OK")
print("Shape:", z.shape)
print("Place:", z.place)

In [ ]:
# Clone PaddleOCR v3.7.0

%cd /content

!rm -rf PaddleOCR

!git clone -q \
    --depth 1 \
    --branch v3.7.0 \
    https://github.com/PaddlePaddle/PaddleOCR.git

%cd /content/PaddleOCR

In [ ]:
!python -m pip install -q -r requirements.txt

In [ ]:
!git describe --tags --always

In [ ]:
!ls configs/rec/PP-OCRv6/

In [ ]:
# Chuẩn hóa label veeff Unicode NFC

import os
import unicodedata

def prepare_label(input_name, output_name):
    input_path = os.path.join(DATA_DIR, input_name)
    output_path = os.path.join(DATA_DIR, output_name)

    output_lines = []
    chars = set()

    missing_images = []
    bad_lines = []

    max_len = 0
    longest_text = ""

    with open(input_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):

            line = line.rstrip("\r\n")

            if not line:
                continue

            parts = line.split("\t", 1)

            if len(parts) != 2:
                bad_lines.append((line_no, line))
                continue

            image_path, text = parts

            # Windows path -> Linux path
            image_path = image_path.replace("\\", "/")

            # Chuẩn hóa tiếng Việt
            text = unicodedata.normalize("NFC", text)

            if os.path.isabs(image_path):
                full_path = image_path
            else:
                full_path = os.path.join(DATA_DIR, image_path)

            if not os.path.exists(full_path):
                missing_images.append(full_path)
                continue

            relative_path = os.path.relpath(
                full_path,
                DATA_DIR
            ).replace("\\", "/")

            output_lines.append(
                f"{relative_path}\t{text}"
            )

            chars.update(text)

            if len(text) > max_len:
                max_len = len(text)
                longest_text = text

    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(output_lines))

    print("=" * 60)
    print("Input :", input_name)
    print("Output:", output_name)
    print("Samples       :", len(output_lines))
    print("Missing images:", len(missing_images))
    print("Bad lines     :", len(bad_lines))
    print("Max text len  :", max_len)
    print("Longest text  :", longest_text)

    return chars, max_len, len(output_lines)

In [ ]:
train_chars, train_max, n_train = prepare_label(
    "train_label.txt",
    "train_v6.txt"
)

valid_chars, valid_max, n_valid = prepare_label(
    "valid_label.txt",
    "valid_v6.txt"
)

test_chars, test_max, n_test = prepare_label(
    "test_label.txt",
    "test_v6.txt"
)

ALL_CHARS = train_chars | valid_chars | test_chars
MAX_DATASET_LENGTH = max(
    train_max,
    valid_max,
    test_max
)

print("\n" + "=" * 60)
print("Train:", n_train)
print("Valid:", n_valid)
print("Test :", n_test)
print("Unique chars:", len(ALL_CHARS))
print("Maximum label length:", MAX_DATASET_LENGTH)

In [ ]:
# Kiểm tra max length

MAX_TEXT_LENGTH = 40

assert MAX_DATASET_LENGTH <= MAX_TEXT_LENGTH, (
    f"Dataset có label dài {MAX_DATASET_LENGTH}, "
    f"lớn hơn MAX_TEXT_LENGTH={MAX_TEXT_LENGTH}"
)

print("MAX_TEXT_LENGTH OK:", MAX_TEXT_LENGTH)

In [ ]:
# Tạo custum Vietnamese dictionary

vietnamese_chars = (
    "aAàÀảẢãÃáÁạẠăĂằẰẳẲẵẴắẮặẶâÂầẦẩẨẫẪấẤậẬ"
    "bBcCdDđĐeEèÈẻẺẽẼéÉẹẸêÊềỀểỂễỄếẾệỆ"
    "fFgGhHiIìÌỉỈĩĨíÍịỊjJkKlLmMnNoO"
    "òÒỏỎõÕóÓọỌôÔồỒổỔỗỖốỐộỘơƠờỜởỞỡỠớỚợỢ"
    "pPqQrRsStTuUùÙủỦũŨúÚụỤưƯừỪửỬữỮứỨựỰ"
    "vVwWxXyYỳỲỷỶỹỸýÝỵỴzZ"
    "0123456789"
    "!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~"
)

# Xóa ký tự trùng nhưng vẫn giữ đúng thứ tự
dict_chars = list(dict.fromkeys(vietnamese_chars))

DICT_PATH = (
    "/content/PaddleOCR/"
    "ppocr/utils/dict/studentcard_vi_dict.txt"
)

with open(DICT_PATH, "w", encoding="utf-8") as f:
    for c in dict_chars:
        f.write(c + "\n")

print("Dictionary size:", len(dict_chars))
print("Saved:", DICT_PATH)

In [ ]:
with open(DICT_PATH, "r", encoding="utf-8") as f:
    custom_chars = {
        line.rstrip("\n")
        for line in f
        if line.rstrip("\n") != ""
    }

# PaddleOCR tự thêm space
custom_chars.add(" ")

missing_chars = sorted(
    c for c in ALL_CHARS
    if c not in custom_chars
)

print("Dataset chars   :", len(ALL_CHARS))
print("Dictionary chars:", len(custom_chars))
print("Missing chars   :", missing_chars)

In [ ]:
assert len(missing_chars) == 0, \
    f"Dictionary còn thiếu: {missing_chars}"

print("Dictionary OK!")

In [ ]:
# Download pretrained PP-OCRv6 small

%cd /content/PaddleOCR

!mkdir -p pretrain_models

!wget -q -nc \
"https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv6_small_rec_pretrained.pdparams" \
-O pretrain_models/PP-OCRv6_small_rec_pretrained.pdparams

In [ ]:
import os

PRETRAINED = (
    "/content/PaddleOCR/pretrain_models/"
    "PP-OCRv6_small_rec_pretrained.pdparams"
)

print("Exists:", os.path.exists(PRETRAINED))

print(
    "Size:",
    round(os.path.getsize(PRETRAINED) / 1024**2, 2),
    "MB"
)

In [ ]:
# Tạo config riêng cho dataset

import yaml
import copy
import os

BASE_CONFIG = (
    "/content/PaddleOCR/"
    "configs/rec/PP-OCRv6/"
    "PP-OCRv6_small_rec.yml"
)

CUSTOM_CONFIG = (
    "/content/PaddleOCR/"
    "configs/rec/PP-OCRv6/"
    "PP-OCRv6_small_studentcard.yml"
)

TRAIN_LABEL = os.path.join(DATA_DIR, "train_v6.txt")
VALID_LABEL = os.path.join(DATA_DIR, "valid_v6.txt")
TEST_LABEL  = os.path.join(DATA_DIR, "test_v6.txt")

with open(BASE_CONFIG, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

In [ ]:
# ============================================================
# GLOBAL
# ============================================================

cfg["Global"]["use_gpu"] = True

cfg["Global"]["epoch_num"] = 50

cfg["Global"]["save_model_dir"] = OUTPUT_DIR

cfg["Global"]["save_epoch_step"] = 5

# ~1 validation / epoch
cfg["Global"]["eval_batch_step"] = [0, 159]

cfg["Global"]["pretrained_model"] = PRETRAINED

cfg["Global"]["checkpoints"] = None

cfg["Global"]["character_dict_path"] = DICT_PATH

cfg["Global"]["max_text_length"] = MAX_TEXT_LENGTH

cfg["Global"]["use_space_char"] = True

cfg["Global"]["distributed"] = False

cfg["Global"]["cal_metric_during_train"] = True


# ============================================================
# OPTIMIZER
# ============================================================

cfg["Optimizer"]["lr"]["learning_rate"] = 5e-4

cfg["Optimizer"]["lr"]["warmup_epoch"] = 5


# ============================================================
# ARCHITECTURE
# ============================================================

for head in cfg["Architecture"]["Head"]["head_list"]:
    if "NRTRHead" in head:
        head["NRTRHead"]["max_text_length"] = MAX_TEXT_LENGTH


# ============================================================
# TRAIN DATA
# ============================================================

cfg["Train"]["dataset"]["data_dir"] = DATA_DIR

cfg["Train"]["dataset"]["label_file_list"] = [
    TRAIN_LABEL
]

for transform in cfg["Train"]["dataset"]["transforms"]:
    if "RecConAug" in transform:
        transform["RecConAug"][
            "max_text_length"
        ] = MAX_TEXT_LENGTH


# ============================================================
# BATCH
# ============================================================

BATCH_SIZE = 128

cfg["Train"]["sampler"]["first_bs"] = BATCH_SIZE

cfg["Train"]["loader"][
    "batch_size_per_card"
] = BATCH_SIZE

cfg["Train"]["loader"]["num_workers"] = 4


# ============================================================
# VALIDATION
# ============================================================

cfg["Eval"]["dataset"]["data_dir"] = DATA_DIR

cfg["Eval"]["dataset"]["label_file_list"] = [
    VALID_LABEL
]

cfg["Eval"]["loader"][
    "batch_size_per_card"
] = BATCH_SIZE

cfg["Eval"]["loader"]["num_workers"] = 4

In [ ]:
# Lưu config
with open(CUSTOM_CONFIG, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        cfg,
        f,
        allow_unicode=True,
        sort_keys=False
    )

print("Saved:", CUSTOM_CONFIG)

In [ ]:
print("Model       :", cfg["Global"]["model_name"])
print("Epoch       :", cfg["Global"]["epoch_num"])
print("LR          :", cfg["Optimizer"]["lr"]["learning_rate"])
print("Train batch :", cfg["Train"]["sampler"]["first_bs"])
print("Max length  :", cfg["Global"]["max_text_length"])
print("Dict        :", cfg["Global"]["character_dict_path"])
print("Train       :", cfg["Train"]["dataset"]["label_file_list"])
print("Valid       :", cfg["Eval"]["dataset"]["label_file_list"])
print("Output      :", cfg["Global"]["save_model_dir"])

In [ ]:
# Lưu dic + config vào drive

import shutil
import os

REPRO_DIR = os.path.join(
    OUTPUT_DIR,
    "reproducibility"
)

os.makedirs(REPRO_DIR, exist_ok=True)

shutil.copy2(
    CUSTOM_CONFIG,
    os.path.join(
        REPRO_DIR,
        "PP-OCRv6_small_studentcard.yml"
    )
)

shutil.copy2(
    DICT_PATH,
    os.path.join(
        REPRO_DIR,
        "studentcard_vi_dict.txt"
    )
)

print("Saved reproducibility files to:")
print(REPRO_DIR)

In [ ]:
# TRAIN

%cd /content/PaddleOCR

!python tools/train.py \
-c configs/rec/PP-OCRv6/PP-OCRv6_small_studentcard.yml

In [ ]:
# Kiểm tra sau khi train

import os

files = sorted(os.listdir(OUTPUT_DIR))

for f in files:
    print(f)

In [ ]:
BEST_MODEL = os.path.join(
    OUTPUT_DIR,
    "best_accuracy.pdparams"
)

print(BEST_MODEL)
print("Exists:", os.path.exists(BEST_MODEL))

In [ ]:
# Đánh giá trên test set

TEST_CONFIG = (
    "/content/PaddleOCR/"
    "configs/rec/PP-OCRv6/"
    "PP-OCRv6_small_studentcard_test.yml"
)

test_cfg = copy.deepcopy(cfg)

test_cfg["Eval"]["dataset"]["data_dir"] = DATA_DIR

test_cfg["Eval"]["dataset"]["label_file_list"] = [
    TEST_LABEL
]

test_cfg["Eval"]["loader"][
    "batch_size_per_card"
] = 64

with open(TEST_CONFIG, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        test_cfg,
        f,
        allow_unicode=True,
        sort_keys=False
    )

print(TEST_CONFIG)

In [ ]:
%cd /content/PaddleOCR

!python tools/eval.py \
-c configs/rec/PP-OCRv6/PP-OCRv6_small_studentcard_test.yml \
-o Global.pretrained_model="{BEST_MODEL}"

In [ ]:
# Test trên một ảnh cụ thể

with open(TEST_LABEL, "r", encoding="utf-8") as f:
    line = f.readline().strip()

image_path, ground_truth = line.split("\t", 1)

TEST_IMAGE = os.path.join(
    DATA_DIR,
    image_path
)

print("Image :", TEST_IMAGE)
print("GT    :", ground_truth)

In [ ]:
%cd /content/PaddleOCR

!python tools/infer_rec.py \
-c configs/rec/PP-OCRv6/PP-OCRv6_small_studentcard.yml \
-o \
Global.pretrained_model="{BEST_MODEL}" \
Global.infer_img="{TEST_IMAGE}"

In [ ]:
# Export inference model

INFERENCE_DIR = os.path.join(
    OUTPUT_DIR,
    "inference"
)

In [ ]:
%cd /content/PaddleOCR

!python tools/export_model.py \
-c configs/rec/PP-OCRv6/PP-OCRv6_small_studentcard.yml \
-o \
Global.pretrained_model="{BEST_MODEL}" \
Global.save_inference_dir="{INFERENCE_DIR}"

In [ ]:
for root, dirs, files in os.walk(INFERENCE_DIR):
    for file in files:
        path = os.path.join(root, file)

        print(
            os.path.relpath(path, INFERENCE_DIR),
            round(
                os.path.getsize(path) / 1024**2,
                3
            ),
            "MB"
        )

In [ ]:
# Kích thước sau khi export

total_size = 0

for root, dirs, files in os.walk(INFERENCE_DIR):

    for file in files:

        path = os.path.join(
            root,
            file
        )

        total_size += os.path.getsize(path)

print(
    "Total inference model size:",
    round(total_size / 1024**2, 2),
    "MB"
)